In [6]:
import os, cv2, torch
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# -------------------------
# Performance
# -------------------------
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# -------------------------
# Config - OPTIMIZED FOR SPEED
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CSV_PATH   = "/kaggle/input/rsna-bone-age/boneage-training-dataset.csv"
PATCH_ROOT = "/kaggle/input/phase1-upd"

# Reduced settings for faster training
IMG_SIZE   = 448      # Reduced from 560 (saves 37% compute)
NUM_BINS   = 240
BATCH_SIZE = 24       # Increased from 16 (better GPU utilization)
EPOCHS     = 40       # Reduced from 120 (1/3 time)
NUM_WORKERS = 2       # Reduced to avoid CPU bottleneck

# Faster learning rate schedule
LR_1 = 5e-4          # Start higher
LR_2 = 2e-4          # Epoch 20
LR_3 = 5e-5          # Epoch 30

# Paper parameters
DELTA = 15.0
LAMBDA = 0.5

# Early stopping
PATIENCE = 8

# -------------------------
# Gaussian Label Distribution (Paper Eq. 11)
# -------------------------
def gaussian_label_distribution(age, delta=DELTA):
    bins = torch.arange(NUM_BINS, device=age.device).float()
    age = age.unsqueeze(1)
    coef = 1.0 / (torch.sqrt(torch.tensor(2 * 3.14159 * delta * delta)))
    exponent = -((bins - age) ** 2) / (2 * delta * delta)
    gaussian = coef * torch.exp(exponent)
    gaussian = gaussian / gaussian.sum(dim=1, keepdim=True)
    return gaussian

def expected_age(logits):
    probs = torch.softmax(logits, dim=1)
    bins = torch.arange(NUM_BINS, device=logits.device).float()
    return (probs * bins).sum(dim=1)

# -------------------------
# Dataset - OPTIMIZED
# -------------------------
class BoneAgeDataset(Dataset):
    def __init__(self, df, mode='H+R1', train=True):  # Changed to H+R1 (faster, almost same accuracy)
        self.df = df.reset_index(drop=True)
        self.mode = mode
        self.train = train

    def load_and_resize_region(self, folder, img_id):
        path = f"{PATCH_ROOT}/{folder}/{img_id}.jpg"
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = int(row.id)
        
        regions = []
        
        if 'H' in self.mode:
            regions.append(self.load_and_resize_region("hand", img_id))
        
        if 'R1' in self.mode:
            regions.append(self.load_and_resize_region("region1", img_id))
            
        if 'R2' in self.mode:
            regions.append(self.load_and_resize_region("region2", img_id))
        
        stacked = np.concatenate(regions, axis=2)
        
        if self.train and np.random.rand() > 0.5:
            stacked = np.fliplr(stacked).copy()
        
        stacked = torch.from_numpy(stacked).permute(2, 0, 1).float() / 255.0
        
        mean = torch.tensor([0.485, 0.456, 0.406]).view(-1, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(-1, 1, 1)
        num_regions = len(regions)
        mean = mean.repeat(num_regions, 1, 1)
        std = std.repeat(num_regions, 1, 1)
        stacked = (stacked - mean) / std
        
        gender = torch.tensor(1.0 if row.male else -1.0, dtype=torch.float32)
        age = torch.tensor(row.boneage, dtype=torch.float32)
        
        return stacked, gender, age

    def __len__(self):
        return len(self.df)

# -------------------------
# Model - LIGHTER VERSION
# -------------------------
class BoneAgeModel(nn.Module):
    def __init__(self, num_input_channels=6):  # 6 for H+R1
        super().__init__()
        
        import timm
        # Use EfficientNet instead of Xception (faster, similar accuracy)
        self.backbone = timm.create_model("tf_efficientnet_b3", pretrained=True, num_classes=0)
        
        # Modify first conv
        if num_input_channels != 3:
            old_conv = self.backbone.conv_stem
            new_conv = nn.Conv2d(
                num_input_channels, old_conv.out_channels,
                kernel_size=old_conv.kernel_size,
                stride=old_conv.stride,
                padding=old_conv.padding,
                bias=False
            )
            with torch.no_grad():
                new_conv.weight[:, :3, :, :] = old_conv.weight
                for i in range(3, num_input_channels):
                    new_conv.weight[:, i:i+1, :, :] = old_conv.weight.mean(dim=1, keepdim=True)
            self.backbone.conv_stem = new_conv
        
        self.gender_fc = nn.Linear(1, 32)
        
        # EfficientNet-B3 outputs 1536 features
        self.fc1 = nn.Linear(1536 + 32, 512)  # Reduced from 1024
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(512, NUM_BINS)

    def forward(self, x, gender):
        features = self.backbone(x)
        gender_feat = self.gender_fc(gender.unsqueeze(1))
        combined = torch.cat([features, gender_feat], dim=1)
        x = self.fc1(combined)
        x = F.relu(x)
        x = self.dropout(x)
        logits = self.fc2(x)
        return logits

# -------------------------
# Loss Function
# -------------------------
def bone_age_loss(logits, age_true, lambda_reg=LAMBDA):
    age_dist_pred = torch.softmax(logits, dim=1)
    age_pred = expected_age(logits)
    l_mae = torch.mean(torch.abs(age_pred - age_true))
    gaussian_dist = gaussian_label_distribution(age_true, delta=DELTA)
    l_reg = F.kl_div(
        torch.log(age_dist_pred + 1e-10),
        gaussian_dist,
        reduction='batchmean'
    )
    total_loss = l_mae + lambda_reg * l_reg
    return total_loss, l_mae, l_reg

# -------------------------
# Training - WITH EARLY STOPPING
# -------------------------
def train_model():
    df = pd.read_csv(CSV_PATH)
    
    print("Filtering valid samples...")
    valid_mask = []
    for _, row in df.iterrows():
        img_id = int(row.id)
        exists = (
            os.path.exists(f"{PATCH_ROOT}/hand/{img_id}.jpg") and
            os.path.exists(f"{PATCH_ROOT}/region1/{img_id}.jpg")
        )
        valid_mask.append(exists)
    
    df = df[valid_mask].reset_index(drop=True)
    print(f"Valid samples: {len(df)}")
    
    # Smaller validation set for speed
    val_df = df.sample(n=500, random_state=42)
    train_df = df.drop(val_df.index)
    
    print(f"Train: {len(train_df)}, Val: {len(val_df)}")
    
    # H+R1 configuration (Paper Table V shows 5.4 MAE, vs 4.8 for H+R1+R2)
    # Small sacrifice for much faster training
    train_dataset = BoneAgeDataset(train_df, mode='H+R1', train=True)
    val_dataset = BoneAgeDataset(val_df, mode='H+R1', train=False)
    
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
    )
    
    # Model with 6 input channels (H+R1)
    model = BoneAgeModel(num_input_channels=6).to(DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_1, weight_decay=0.01)
    scaler = GradScaler()
    
    best_mae = float('inf')
    patience_counter = 0
    
    print(f"\n{'='*60}")
    print(f"FAST TRAINING MODE")
    print(f"Image Size: {IMG_SIZE}x{IMG_SIZE} (vs 560x560 in paper)")
    print(f"Model: EfficientNet-B3 (vs Xception in paper)")
    print(f"Regions: H+R1 (vs H+R1+R2 in paper)")
    print(f"Epochs: {EPOCHS} (vs 120 in paper)")
    print(f"Expected: ~5-6 months MAE in ~{EPOCHS*10} minutes")
    print(f"Paper result: 5.4 months MAE with H+R1")
    print(f"{'='*60}\n")
    
    for epoch in range(EPOCHS):
        # Learning rate schedule
        if epoch == 20:
            for param_group in optimizer.param_groups:
                param_group['lr'] = LR_2
            print(f"Learning rate → {LR_2}")
        elif epoch == 30:
            for param_group in optimizer.param_groups:
                param_group['lr'] = LR_3
            print(f"Learning rate → {LR_3}")
        
        # Train
        model.train()
        train_loss = 0
        train_mae = 0
        
        for images, gender, age in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            images = images.to(DEVICE)
            gender = gender.to(DEVICE)
            age = age.to(DEVICE)
            
            optimizer.zero_grad()
            
            with autocast():
                logits = model(images, gender)
                loss, mae, reg = bone_age_loss(logits, age, lambda_reg=LAMBDA)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item()
            train_mae += mae.item()
        
        train_loss /= len(train_loader)
        train_mae /= len(train_loader)
        
        # Validation
        model.eval()
        val_mae = 0
        
        with torch.no_grad():
            for images, gender, age in val_loader:
                images = images.to(DEVICE)
                gender = gender.to(DEVICE)
                age = age.to(DEVICE)
                
                logits = model(images, gender)
                pred_age = expected_age(logits)
                val_mae += torch.mean(torch.abs(pred_age - age)).item()
        
        val_mae /= len(val_loader)
        
        print(f"Epoch {epoch+1}: Loss={train_loss:.4f}, Train MAE={train_mae:.3f}, Val MAE={val_mae:.3f}")
        
        # Save best model
        if val_mae < best_mae:
            best_mae = val_mae
            patience_counter = 0
            torch.save(model.state_dict(), "best_model_fast.pt")
            print(f"✓ Best model saved! MAE: {best_mae:.3f}")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break
    
    print(f"\n{'='*60}")
    print(f"Training complete!")
    print(f"Best validation MAE: {best_mae:.3f} months")
    print(f"Paper H+R1 result: 5.4 months")
    print(f"Paper H+R1+R2 result: 4.8 months")
    print(f"{'='*60}")

if __name__ == "__main__":
    train_model()

Filtering valid samples...
Valid samples: 12585
Train: 12085, Val: 500


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

/tmp/ipykernel_55/3298198163.py:218: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()



FAST TRAINING MODE
Image Size: 448x448 (vs 560x560 in paper)
Model: EfficientNet-B3 (vs Xception in paper)
Regions: H+R1 (vs H+R1+R2 in paper)
Epochs: 40 (vs 120 in paper)
Expected: ~5-6 months MAE in ~400 minutes
Paper result: 5.4 months MAE with H+R1



Epoch 1/40:   0%|          | 0/504 [00:00<?, ?it/s]/tmp/ipykernel_55/3298198163.py:256: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1/40: 100%|██████████| 504/504 [05:27<00:00,  1.54it/s]


...

Epoch 20: Loss=4.9715, Train MAE=4.863, Val MAE=7.492
Learning rate → 0.0002


Epoch 21/40: 100%|██████████| 504/504 [04:20<00:00,  1.94it/s]


Epoch 21: Loss=4.2455, Train MAE=4.156, Val MAE=6.970


Epoch 22/40: 100%|██████████| 504/504 [04:20<00:00,  1.94it/s]


Epoch 22: Loss=3.8029, Train MAE=3.721, Val MAE=6.863

Early stopping at epoch 22

Training complete!
Best validation MAE: 6.579 months
Paper H+R1 result: 5.4 months
Paper H+R1+R2 result: 4.8 months


In [7]:
import os, cv2, torch
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# -------------------------
# Performance
# -------------------------
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# -------------------------
# Config - SAME AS BEFORE
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CSV_PATH   = "/kaggle/input/rsna-bone-age/boneage-training-dataset.csv"
PATCH_ROOT = "/kaggle/input/phase1-upd"

IMG_SIZE   = 448
NUM_BINS   = 240
BATCH_SIZE = 24
NUM_WORKERS = 2

# Resume from epoch 22, train until epoch 60
START_EPOCH = 22
TOTAL_EPOCHS = 60  # Train for 38 more epochs

# Learning rate schedule
LR_1 = 2e-4  # Current LR (was already reduced at epoch 20)
LR_2 = 5e-5  # Drop at epoch 40
LR_3 = 1e-5  # Drop at epoch 50

DELTA = 15.0
LAMBDA = 0.5

# -------------------------
# Gaussian Label Distribution
# -------------------------
def gaussian_label_distribution(age, delta=DELTA):
    bins = torch.arange(NUM_BINS, device=age.device).float()
    age = age.unsqueeze(1)
    coef = 1.0 / (torch.sqrt(torch.tensor(2 * 3.14159 * delta * delta)))
    exponent = -((bins - age) ** 2) / (2 * delta * delta)
    gaussian = coef * torch.exp(exponent)
    gaussian = gaussian / gaussian.sum(dim=1, keepdim=True)
    return gaussian

def expected_age(logits):
    probs = torch.softmax(logits, dim=1)
    bins = torch.arange(NUM_BINS, device=logits.device).float()
    return (probs * bins).sum(dim=1)

# -------------------------
# Dataset - H+R1 (same as before)
# -------------------------
class BoneAgeDataset(Dataset):
    def __init__(self, df, mode='H+R1', train=True):
        self.df = df.reset_index(drop=True)
        self.mode = mode
        self.train = train

    def load_and_resize_region(self, folder, img_id):
        path = f"{PATCH_ROOT}/{folder}/{img_id}.jpg"
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = int(row.id)
        
        regions = []
        
        if 'H' in self.mode:
            regions.append(self.load_and_resize_region("hand", img_id))
        
        if 'R1' in self.mode:
            regions.append(self.load_and_resize_region("region1", img_id))
            
        if 'R2' in self.mode:
            regions.append(self.load_and_resize_region("region2", img_id))
        
        stacked = np.concatenate(regions, axis=2)
        
        if self.train and np.random.rand() > 0.5:
            stacked = np.fliplr(stacked).copy()
        
        stacked = torch.from_numpy(stacked).permute(2, 0, 1).float() / 255.0
        
        mean = torch.tensor([0.485, 0.456, 0.406]).view(-1, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(-1, 1, 1)
        num_regions = len(regions)
        mean = mean.repeat(num_regions, 1, 1)
        std = std.repeat(num_regions, 1, 1)
        stacked = (stacked - mean) / std
        
        gender = torch.tensor(1.0 if row.male else -1.0, dtype=torch.float32)
        age = torch.tensor(row.boneage, dtype=torch.float32)
        
        return stacked, gender, age

    def __len__(self):
        return len(self.df)

# -------------------------
# Model - EfficientNet-B3 (same as before)
# -------------------------
class BoneAgeModel(nn.Module):
    def __init__(self, num_input_channels=6):
        super().__init__()
        
        import timm
        self.backbone = timm.create_model("tf_efficientnet_b3", pretrained=True, num_classes=0)
        
        if num_input_channels != 3:
            old_conv = self.backbone.conv_stem
            new_conv = nn.Conv2d(
                num_input_channels, old_conv.out_channels,
                kernel_size=old_conv.kernel_size,
                stride=old_conv.stride,
                padding=old_conv.padding,
                bias=False
            )
            with torch.no_grad():
                new_conv.weight[:, :3, :, :] = old_conv.weight
                for i in range(3, num_input_channels):
                    new_conv.weight[:, i:i+1, :, :] = old_conv.weight.mean(dim=1, keepdim=True)
            self.backbone.conv_stem = new_conv
        
        self.gender_fc = nn.Linear(1, 32)
        self.fc1 = nn.Linear(1536 + 32, 512)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(512, NUM_BINS)

    def forward(self, x, gender):
        features = self.backbone(x)
        gender_feat = self.gender_fc(gender.unsqueeze(1))
        combined = torch.cat([features, gender_feat], dim=1)
        x = self.fc1(combined)
        x = F.relu(x)
        x = self.dropout(x)
        logits = self.fc2(x)
        return logits

# -------------------------
# Loss Function
# -------------------------
def bone_age_loss(logits, age_true, lambda_reg=LAMBDA):
    age_dist_pred = torch.softmax(logits, dim=1)
    age_pred = expected_age(logits)
    l_mae = torch.mean(torch.abs(age_pred - age_true))
    gaussian_dist = gaussian_label_distribution(age_true, delta=DELTA)
    l_reg = F.kl_div(
        torch.log(age_dist_pred + 1e-10),
        gaussian_dist,
        reduction='batchmean'
    )
    total_loss = l_mae + lambda_reg * l_reg
    return total_loss, l_mae, l_reg

# -------------------------
# Resume Training
# -------------------------
def resume_training():
    df = pd.read_csv(CSV_PATH)
    
    print("Filtering valid samples...")
    valid_mask = []
    for _, row in df.iterrows():
        img_id = int(row.id)
        exists = (
            os.path.exists(f"{PATCH_ROOT}/hand/{img_id}.jpg") and
            os.path.exists(f"{PATCH_ROOT}/region1/{img_id}.jpg")
        )
        valid_mask.append(exists)
    
    df = df[valid_mask].reset_index(drop=True)
    print(f"Valid samples: {len(df)}")
    
    # Same split as before (IMPORTANT: use same random_state!)
    val_df = df.sample(n=500, random_state=42)
    train_df = df.drop(val_df.index)
    
    print(f"Train: {len(train_df)}, Val: {len(val_df)}")
    
    # Same configuration: H+R1
    train_dataset = BoneAgeDataset(train_df, mode='H+R1', train=True)
    val_dataset = BoneAgeDataset(val_df, mode='H+R1', train=False)
    
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
    )
    
    # Load the saved model
    print("\n" + "="*60)
    print("RESUMING TRAINING FROM CHECKPOINT")
    print("="*60)
    
    model = BoneAgeModel(num_input_channels=6).to(DEVICE)
    
    # Load the best checkpoint
    checkpoint_path = "best_model_fast.pt"
    if os.path.exists(checkpoint_path):
        model.load_state_dict(torch.load(checkpoint_path))
        print(f"✓ Loaded checkpoint: {checkpoint_path}")
        print(f"✓ Previous best MAE: 6.579 months")
    else:
        print("ERROR: Checkpoint not found! Make sure 'best_model_fast.pt' exists.")
        return
    
    print(f"✓ Resuming from epoch {START_EPOCH}")
    print(f"✓ Training until epoch {TOTAL_EPOCHS}")
    print(f"✓ Current LR: {LR_1}")
    print(f"✓ Will drop to {LR_2} at epoch 40")
    print(f"✓ Will drop to {LR_3} at epoch 50")
    print("="*60 + "\n")
    
    # Optimizer with current learning rate
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_1, weight_decay=0.01)
    scaler = GradScaler()
    
    best_mae = 6.579  # Starting from previous best
    
    for epoch in range(START_EPOCH, TOTAL_EPOCHS):
        # Learning rate schedule
        if epoch == 40:
            for param_group in optimizer.param_groups:
                param_group['lr'] = LR_2
            print(f"\n>>> Learning rate → {LR_2}\n")
        elif epoch == 50:
            for param_group in optimizer.param_groups:
                param_group['lr'] = LR_3
            print(f"\n>>> Learning rate → {LR_3}\n")
        
        # Train
        model.train()
        train_loss = 0
        train_mae = 0
        
        for images, gender, age in tqdm(train_loader, desc=f"Epoch {epoch+1}/{TOTAL_EPOCHS}"):
            images = images.to(DEVICE)
            gender = gender.to(DEVICE)
            age = age.to(DEVICE)
            
            optimizer.zero_grad()
            
            with autocast():
                logits = model(images, gender)
                loss, mae, reg = bone_age_loss(logits, age, lambda_reg=LAMBDA)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item()
            train_mae += mae.item()
        
        train_loss /= len(train_loader)
        train_mae /= len(train_loader)
        
        # Validation
        model.eval()
        val_mae = 0
        
        with torch.no_grad():
            for images, gender, age in val_loader:
                images = images.to(DEVICE)
                gender = gender.to(DEVICE)
                age = age.to(DEVICE)
                
                logits = model(images, gender)
                pred_age = expected_age(logits)
                val_mae += torch.mean(torch.abs(pred_age - age)).item()
        
        val_mae /= len(val_loader)
        
        print(f"Epoch {epoch+1}: Loss={train_loss:.4f}, Train MAE={train_mae:.3f}, Val MAE={val_mae:.3f}")
        
        # Save if better
        if val_mae < best_mae:
            best_mae = val_mae
            torch.save(model.state_dict(), "best_model_resumed.pt")
            print(f"✓ NEW BEST! Saved model with MAE: {best_mae:.3f}")
    
    print(f"\n{'='*60}")
    print(f"TRAINING COMPLETE!")
    print(f"Best validation MAE: {best_mae:.3f} months")
    print(f"Target (Paper H+R1): 5.4 months")
    print(f"Gap: {best_mae - 5.4:.3f} months")
    print(f"{'='*60}")

if __name__ == "__main__":
    resume_training()

Filtering valid samples...
Valid samples: 12585
Train: 12085, Val: 500

RESUMING TRAINING FROM CHECKPOINT


/tmp/ipykernel_55/3291565135.py:233: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


✓ Loaded checkpoint: best_model_fast.pt
✓ Previous best MAE: 6.579 months
✓ Resuming from epoch 22
✓ Training until epoch 60
✓ Current LR: 0.0002
✓ Will drop to 5e-05 at epoch 40
✓ Will drop to 1e-05 at epoch 50



Epoch 23/60:   0%|          | 0/504 [00:00<?, ?it/s]/tmp/ipykernel_55/3291565135.py:260: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 23/60: 100%|██████████| 504/504 [04:20<00:00,  1.94it/s]


Epoch 23: Loss=5.1399, Train MAE=5.038, Val MAE=6.489
✓ NEW BEST! Saved model with MAE: 6.489


...

Epoch 58: Loss=1.1378, Train MAE=1.078, Val MAE=6.222
✓ NEW BEST! Saved model with MAE: 6.222


Epoch 59/60: 100%|██████████| 504/504 [04:20<00:00,  1.93it/s]


Epoch 59: Loss=1.1242, Train MAE=1.064, Val MAE=6.322


Epoch 60/60: 100%|██████████| 504/504 [04:20<00:00,  1.93it/s]


Epoch 60: Loss=1.1065, Train MAE=1.047, Val MAE=6.271

TRAINING COMPLETE!
Best validation MAE: 6.222 months
Target (Paper H+R1): 5.4 months
Gap: 0.822 months


In [10]:
import os, cv2, torch
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# -------------------------
# Config
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 448
NUM_BINS = 240
BATCH_SIZE = 24
NUM_WORKERS = 2

# -------------------------
# Your Dataset and Model (same as before)
# -------------------------
class BoneAgeDataset(Dataset):
    def __init__(self, df, patch_root, mode='H+R1', train=False):
        self.df = df.reset_index(drop=True)
        self.patch_root = patch_root
        self.mode = mode
        self.train = train

    def load_and_resize_region(self, folder, img_id):
        path = f"{self.patch_root}/{folder}/{img_id}.jpg"
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = int(row.id)
        
        regions = []
        if 'H' in self.mode:
            regions.append(self.load_and_resize_region("hand", img_id))
        if 'R1' in self.mode:
            regions.append(self.load_and_resize_region("region1", img_id))
        if 'R2' in self.mode:
            regions.append(self.load_and_resize_region("region2", img_id))
        
        stacked = np.concatenate(regions, axis=2)
        stacked = torch.from_numpy(stacked).permute(2, 0, 1).float() / 255.0
        
        mean = torch.tensor([0.485, 0.456, 0.406]).view(-1, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(-1, 1, 1)
        num_regions = len(regions)
        mean = mean.repeat(num_regions, 1, 1)
        std = std.repeat(num_regions, 1, 1)
        stacked = (stacked - mean) / std
        
        gender = torch.tensor(1.0 if row.male else -1.0, dtype=torch.float32)
        age = torch.tensor(row.boneage if 'boneage' in row else 0.0, dtype=torch.float32)
        
        return stacked, gender, age, img_id

    def __len__(self):
        return len(self.df)

class BoneAgeModel(nn.Module):
    def __init__(self, num_input_channels=6):
        super().__init__()
        import timm
        self.backbone = timm.create_model("tf_efficientnet_b3", pretrained=True, num_classes=0)
        
        if num_input_channels != 3:
            old_conv = self.backbone.conv_stem
            new_conv = nn.Conv2d(
                num_input_channels, old_conv.out_channels,
                kernel_size=old_conv.kernel_size,
                stride=old_conv.stride,
                padding=old_conv.padding,
                bias=False
            )
            with torch.no_grad():
                new_conv.weight[:, :3, :, :] = old_conv.weight
                for i in range(3, num_input_channels):
                    new_conv.weight[:, i:i+1, :, :] = old_conv.weight.mean(dim=1, keepdim=True)
            self.backbone.conv_stem = new_conv
        
        self.gender_fc = nn.Linear(1, 32)
        self.fc1 = nn.Linear(1536 + 32, 512)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(512, NUM_BINS)

    def forward(self, x, gender):
        features = self.backbone(x)
        gender_feat = self.gender_fc(gender.unsqueeze(1))
        combined = torch.cat([features, gender_feat], dim=1)
        x = self.fc1(combined)
        x = F.relu(x)
        x = self.dropout(x)
        logits = self.fc2(x)
        return logits

def expected_age(logits):
    probs = torch.softmax(logits, dim=1)
    bins = torch.arange(NUM_BINS, device=logits.device).float()
    return (probs * bins).sum(dim=1)

# -------------------------
# TTA Inference (BEST METHOD)
# -------------------------
def predict_with_tta(model, images, gender):
    """
    Test-Time Augmentation with horizontal flip.
    This is the best post-processing method (5.302 MAE).
    """
    model.eval()
    
    with torch.no_grad():
        # Original prediction
        logits_orig = model(images, gender)
        pred_orig = expected_age(logits_orig)
        
        # Flipped prediction
        images_flip = torch.flip(images, dims=[3])
        logits_flip = model(images_flip, gender)
        pred_flip = expected_age(logits_flip)
        
        # Average both predictions
        pred_final = (pred_orig + pred_flip) / 2.0
    
    return pred_final

# -------------------------
# Inference Functions
# -------------------------
def make_predictions(model, dataloader, use_tta=True):
    """
    Generate predictions for a dataset.
    
    Args:
        model: Trained BoneAgeModel
        dataloader: DataLoader with test data
        use_tta: Whether to use Test-Time Augmentation (recommended: True)
    
    Returns:
        Dictionary with predictions and metadata
    """
    model.eval()
    
    all_predictions = []
    all_actuals = []
    all_ids = []
    all_genders = []
    
    with torch.no_grad():
        for images, gender, age, img_id in tqdm(dataloader, desc="Predicting"):
            images = images.to(DEVICE)
            gender = gender.to(DEVICE)
            
            if use_tta:
                pred_age = predict_with_tta(model, images, gender)
            else:
                logits = model(images, gender)
                pred_age = expected_age(logits)
            
            all_predictions.extend(pred_age.cpu().numpy())
            all_actuals.extend(age.cpu().numpy())
            all_ids.extend(img_id.cpu().numpy())
            all_genders.extend(gender.cpu().numpy())
    
    return {
        'id': all_ids,
        'predicted_age': all_predictions,
        'actual_age': all_actuals,
        'gender': all_genders
    }

def evaluate_predictions(predictions_dict):
    """Calculate evaluation metrics."""
    preds = np.array(predictions_dict['predicted_age'])
    actuals = np.array(predictions_dict['actual_age'])
    
    mae = np.mean(np.abs(preds - actuals))
    rmse = np.sqrt(np.mean((preds - actuals) ** 2))
    median_ae = np.median(np.abs(preds - actuals))
    
    return {
        'MAE': mae,
        'RMSE': rmse,
        'Median AE': median_ae
    }

# -------------------------
# Example Usage
# -------------------------
def run_inference_example():
    """
    Example of how to use this for inference.
    """
    print("="*60)
    print("BONE AGE PREDICTION - INFERENCE WITH TTA")
    print("="*60)
    
    # Load your test data
    CSV_PATH = "/kaggle/input/rsna-bone-age/boneage-training-dataset.csv"
    PATCH_ROOT = "/kaggle/input/phase1-upd"
    
    df = pd.read_csv(CSV_PATH)
    
    # Filter valid samples
    valid_mask = []
    for _, row in df.iterrows():
        img_id = int(row.id)
        exists = (
            os.path.exists(f"{PATCH_ROOT}/hand/{img_id}.jpg") and
            os.path.exists(f"{PATCH_ROOT}/region1/{img_id}.jpg")
        )
        valid_mask.append(exists)
    df = df[valid_mask].reset_index(drop=True)
    
    # Use a test set (or all data for submission)
    test_df = df.sample(n=500, random_state=999)
    
    test_dataset = BoneAgeDataset(test_df, PATCH_ROOT, mode='H+R1', train=False)
    test_loader = DataLoader(
        test_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False,
        num_workers=NUM_WORKERS, 
        pin_memory=True
    )
    
    # Load model
    print("\nLoading model...")
    model = BoneAgeModel(num_input_channels=6).to(DEVICE)
    model.load_state_dict(torch.load("best_model_fast.pt"))
    print("✓ Model loaded")
    
    # Make predictions with TTA (BEST METHOD)
    print("\nGenerating predictions with TTA...")
    results = make_predictions(model, test_loader, use_tta=True)
    
    # Evaluate
    metrics = evaluate_predictions(results)
    print(f"\nResults:")
    print(f"  MAE:       {metrics['MAE']:.3f} months")
    print(f"  RMSE:      {metrics['RMSE']:.3f} months")
    print(f"  Median AE: {metrics['Median AE']:.3f} months")
    print(f"\n  Target:    5.400 months")
    print(f"  Gap:       {metrics['MAE'] - 5.4:+.3f} months")
    
    # Save predictions
    results_df = pd.DataFrame({
        'id': results['id'],
        'predicted_boneage': results['predicted_age'],
        'actual_boneage': results['actual_age'],
        'male': (np.array(results['gender']) > 0).astype(int)
    })
    
    results_df.to_csv('predictions_with_tta.csv', index=False)
    print(f"\n✓ Predictions saved to 'predictions_with_tta.csv'")
    print("="*60)
    
    return results_df

if __name__ == "__main__":
    run_inference_example()

BONE AGE PREDICTION - INFERENCE WITH TTA

Loading model...
✓ Model loaded

Generating predictions with TTA...


Predicting: 100%|██████████| 21/21 [00:09<00:00,  2.24it/s]


Results:
  MAE:       5.302 months
  RMSE:      7.589 months
  Median AE: 3.704 months

  Target:    5.400 months
  Gap:       -0.098 months

✓ Predictions saved to 'predictions_with_tta.csv'


In [1]:
import os, cv2, torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.isotonic import IsotonicRegression
import torch.nn.functional as F
from scipy.ndimage import rotate as scipy_rotate

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# -------------------------
# Same config as your code
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CSV_PATH   = "/kaggle/input/rsna-bone-age/boneage-training-dataset.csv"
PATCH_ROOT = "/kaggle/input/phase1-upd"
IMG_SIZE   = 448
NUM_BINS   = 240
BATCH_SIZE = 24
NUM_WORKERS = 2

# -------------------------
# Copy your Dataset and Model classes
# -------------------------
class BoneAgeDataset(Dataset):
    def __init__(self, df, mode='H+R1', train=True):
        self.df = df.reset_index(drop=True)
        self.mode = mode
        self.train = train

    def load_and_resize_region(self, folder, img_id):
        path = f"{PATCH_ROOT}/{folder}/{img_id}.jpg"
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = int(row.id)
        
        regions = []
        if 'H' in self.mode:
            regions.append(self.load_and_resize_region("hand", img_id))
        if 'R1' in self.mode:
            regions.append(self.load_and_resize_region("region1", img_id))
        if 'R2' in self.mode:
            regions.append(self.load_and_resize_region("region2", img_id))
        
        stacked = np.concatenate(regions, axis=2)
        
        if self.train and np.random.rand() > 0.5:
            stacked = np.fliplr(stacked).copy()
        
        stacked = torch.from_numpy(stacked).permute(2, 0, 1).float() / 255.0
        
        mean = torch.tensor([0.485, 0.456, 0.406]).view(-1, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(-1, 1, 1)
        num_regions = len(regions)
        mean = mean.repeat(num_regions, 1, 1)
        std = std.repeat(num_regions, 1, 1)
        stacked = (stacked - mean) / std
        
        gender = torch.tensor(1.0 if row.male else -1.0, dtype=torch.float32)
        age = torch.tensor(row.boneage, dtype=torch.float32)
        
        return stacked, gender, age

    def __len__(self):
        return len(self.df)

class BoneAgeModel(nn.Module):
    def __init__(self, num_input_channels=6):
        super().__init__()
        import timm
        self.backbone = timm.create_model("tf_efficientnet_b3", pretrained=True, num_classes=0)
        
        if num_input_channels != 3:
            old_conv = self.backbone.conv_stem
            new_conv = nn.Conv2d(
                num_input_channels, old_conv.out_channels,
                kernel_size=old_conv.kernel_size,
                stride=old_conv.stride,
                padding=old_conv.padding,
                bias=False
            )
            with torch.no_grad():
                new_conv.weight[:, :3, :, :] = old_conv.weight
                for i in range(3, num_input_channels):
                    new_conv.weight[:, i:i+1, :, :] = old_conv.weight.mean(dim=1, keepdim=True)
            self.backbone.conv_stem = new_conv
        
        self.gender_fc = nn.Linear(1, 32)
        self.fc1 = nn.Linear(1536 + 32, 512)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(512, NUM_BINS)

    def forward(self, x, gender):
        features = self.backbone(x)
        gender_feat = self.gender_fc(gender.unsqueeze(1))
        combined = torch.cat([features, gender_feat], dim=1)
        x = self.fc1(combined)
        x = F.relu(x)
        x = self.dropout(x)
        logits = self.fc2(x)
        return logits

def expected_age(logits):
    probs = torch.softmax(logits, dim=1)
    bins = torch.arange(NUM_BINS, device=logits.device).float()
    return (probs * bins).sum(dim=1)

# -------------------------
# FIXED: Simpler TTA without rotation/brightness
# -------------------------
def predict_with_tta(model, images, gender):
    """Simple TTA with just horizontal flip."""
    model.eval()
    predictions = []
    
    with torch.no_grad():
        # Original
        logits = model(images, gender)
        predictions.append(expected_age(logits))
        
        # Horizontal flip only (most reliable)
        images_flip = torch.flip(images, dims=[3])
        logits = model(images_flip, gender)
        predictions.append(expected_age(logits))
    
    return torch.stack(predictions).mean(dim=0)

# -------------------------
# FIXED: Only simple calibrations
# -------------------------
class SimpleGenderCalibrator:
    """Simple mean bias correction per gender."""
    def __init__(self):
        self.male_bias = 0.0
        self.female_bias = 0.0
    
    def fit(self, predictions, actuals, genders):
        """Calculate biases from arrays."""
        predictions = np.array(predictions)
        actuals = np.array(actuals)
        genders = np.array(genders)
        
        male_mask = genders > 0
        female_mask = genders < 0
        
        if male_mask.sum() > 0:
            self.male_bias = np.mean(predictions[male_mask] - actuals[male_mask])
        if female_mask.sum() > 0:
            self.female_bias = np.mean(predictions[female_mask] - actuals[female_mask])
        
        print(f"  Gender biases - Male: {self.male_bias:.3f}, Female: {self.female_bias:.3f}")
    
    def apply(self, pred_age, gender):
        """Apply correction."""
        if isinstance(pred_age, torch.Tensor):
            correction = torch.where(
                gender > 0,
                torch.tensor(self.male_bias, device=pred_age.device),
                torch.tensor(self.female_bias, device=pred_age.device)
            )
            return pred_age - correction
        else:
            correction = np.where(gender > 0, self.male_bias, self.female_bias)
            return pred_age - correction

class SimpleIsotonicCalibrator:
    """Isotonic regression."""
    def __init__(self):
        self.calibrator = IsotonicRegression(out_of_bounds='clip')
        self.fitted = False
    
    def fit(self, predictions, actuals):
        """Train calibrator."""
        predictions = np.array(predictions)
        actuals = np.array(actuals)
        self.calibrator.fit(predictions, actuals)
        self.fitted = True
        print(f"  Isotonic calibrator fitted on {len(predictions)} samples")
    
    def apply(self, pred_age):
        """Apply calibration."""
        if not self.fitted:
            return pred_age
        
        if isinstance(pred_age, torch.Tensor):
            pred_np = pred_age.cpu().numpy()
            calibrated = self.calibrator.predict(pred_np)
            return torch.tensor(calibrated, device=pred_age.device, dtype=pred_age.dtype)
        else:
            return self.calibrator.predict(pred_age)

# -------------------------
# FIXED: Proper train/calibration/test split
# -------------------------
def main():
    print("Loading data...")
    df = pd.read_csv(CSV_PATH)
    
    # Filter valid samples
    valid_mask = []
    for _, row in df.iterrows():
        img_id = int(row.id)
        exists = (
            os.path.exists(f"{PATCH_ROOT}/hand/{img_id}.jpg") and
            os.path.exists(f"{PATCH_ROOT}/region1/{img_id}.jpg")
        )
        valid_mask.append(exists)
    
    df = df[valid_mask].reset_index(drop=True)
    print(f"Valid samples: {len(df)}")
    
    # Use YOUR ORIGINAL validation split (random_state=42)
    # This is what you've been using during training
    val_df = df.sample(n=500, random_state=42)
    train_df = df.drop(val_df.index)
    
    # For calibration: use a small subset of training data
    # This way we can calibrate AND test on your original val set
    calibration_df = train_df.sample(n=500, random_state=777)
    
    print(f"\nData splits:")
    print(f"  Train: {len(train_df)}")
    print(f"  Calibration (from train): {len(calibration_df)}")
    print(f"  Test (YOUR val set): {len(val_df)}")
    
    # Create datasets
    calib_dataset = BoneAgeDataset(calibration_df, mode='H+R1', train=False)
    val_dataset = BoneAgeDataset(val_df, mode='H+R1', train=False)  # YOUR val set
    
    calib_loader = DataLoader(
        calib_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True
    )
    
    # Load model
    print("\nLoading model...")
    model = BoneAgeModel(num_input_channels=6).to(DEVICE)
    checkpoint_path = "best_model_fast.pt"
    
    if os.path.exists(checkpoint_path):
        model.load_state_dict(torch.load(checkpoint_path))
        print(f"✓ Loaded: {checkpoint_path}")
    else:
        print("ERROR: Model checkpoint not found!")
        return
    
    model.eval()
    
    # -------------------------
    # Step 1: Get calibration predictions
    # -------------------------
    print("\n" + "="*60)
    print("STEP 1: Collecting calibration set predictions")
    print("="*60)
    
    calib_preds = []
    calib_actuals = []
    calib_genders = []
    
    with torch.no_grad():
        for images, gender, age in tqdm(calib_loader, desc="Calibration predictions"):
            images = images.to(DEVICE)
            gender_cpu = gender.cpu().numpy()
            
            logits = model(images, gender.to(DEVICE))
            pred_age = expected_age(logits).cpu().numpy()
            
            calib_preds.extend(pred_age)
            calib_actuals.extend(age.cpu().numpy())
            calib_genders.extend(gender_cpu)
    
    calib_preds = np.array(calib_preds)
    calib_actuals = np.array(calib_actuals)
    calib_genders = np.array(calib_genders)
    
    baseline_calib_mae = np.mean(np.abs(calib_preds - calib_actuals))
    print(f"Calibration set baseline MAE: {baseline_calib_mae:.3f}")
    
    # -------------------------
    # Step 2: Fit calibrators on calibration set
    # -------------------------
    print("\n" + "="*60)
    print("STEP 2: Training calibrators on calibration set")
    print("="*60)
    
    gender_cal = SimpleGenderCalibrator()
    gender_cal.fit(calib_preds, calib_actuals, calib_genders)
    
    isotonic_cal = SimpleIsotonicCalibrator()
    isotonic_cal.fit(calib_preds, calib_actuals)
    
    # -------------------------
    # Step 3: Evaluate on YOUR VALIDATION set
    # -------------------------
    print("\n" + "="*60)
    print("STEP 3: Testing on YOUR validation set (random_state=42)")
    print("="*60)
    
    results = {}
    
    # Baseline (no post-processing)
    print("\n[1/4] Baseline...")
    test_mae = 0.0
    num_samples = 0
    with torch.no_grad():
        for images, gender, age in tqdm(val_loader, desc="Baseline"):
            images = images.to(DEVICE)
            logits = model(images, gender.to(DEVICE))
            pred_age = expected_age(logits)
            test_mae += torch.sum(torch.abs(pred_age - age.to(DEVICE))).item()
            num_samples += len(age)
    results['Baseline'] = test_mae / num_samples
    print(f"  MAE: {results['Baseline']:.3f} months")
    
    # TTA
    print("\n[2/4] TTA (flip only)...")
    test_mae = 0.0
    num_samples = 0
    with torch.no_grad():
        for images, gender, age in tqdm(val_loader, desc="TTA"):
            images = images.to(DEVICE)
            gender = gender.to(DEVICE)
            pred_age = predict_with_tta(model, images, gender)
            test_mae += torch.sum(torch.abs(pred_age - age.to(DEVICE))).item()
            num_samples += len(age)
    results['TTA'] = test_mae / num_samples
    print(f"  MAE: {results['TTA']:.3f} months")
    
    # Gender calibration
    print("\n[3/4] Gender calibration...")
    test_mae = 0.0
    num_samples = 0
    with torch.no_grad():
        for images, gender, age in tqdm(val_loader, desc="Gender Cal"):
            images = images.to(DEVICE)
            gender_dev = gender.to(DEVICE)
            logits = model(images, gender_dev)
            pred_age = expected_age(logits)
            pred_age = gender_cal.apply(pred_age, gender_dev)
            test_mae += torch.sum(torch.abs(pred_age - age.to(DEVICE))).item()
            num_samples += len(age)
    results['Gender Cal'] = test_mae / num_samples
    print(f"  MAE: {results['Gender Cal']:.3f} months")
    
    # Isotonic calibration
    print("\n[4/4] Isotonic calibration...")
    test_mae = 0.0
    num_samples = 0
    with torch.no_grad():
        for images, gender, age in tqdm(val_loader, desc="Isotonic"):
            images = images.to(DEVICE)
            logits = model(images, gender.to(DEVICE))
            pred_age = expected_age(logits)
            pred_age = isotonic_cal.apply(pred_age)
            test_mae += torch.sum(torch.abs(pred_age - age.to(DEVICE))).item()
            num_samples += len(age)
    results['Isotonic'] = test_mae / num_samples
    print(f"  MAE: {results['Isotonic']:.3f} months")
    
    # Combined: TTA + Gender + Isotonic
    print("\n[5/5] Combined (TTA + Gender + Isotonic)...")
    test_mae = 0.0
    num_samples = 0
    with torch.no_grad():
        for images, gender, age in tqdm(val_loader, desc="Combined"):
            images = images.to(DEVICE)
            gender_dev = gender.to(DEVICE)
            
            # TTA
            pred_age = predict_with_tta(model, images, gender_dev)
            # Gender calibration
            pred_age = gender_cal.apply(pred_age, gender_dev)
            # Isotonic calibration
            pred_age = isotonic_cal.apply(pred_age)
            
            test_mae += torch.sum(torch.abs(pred_age - age.to(DEVICE))).item()
            num_samples += len(age)
    results['Combined'] = test_mae / num_samples
    print(f"  MAE: {results['Combined']:.3f} months")
    
    # -------------------------
    # Final Summary
    # -------------------------
    print("\n" + "="*60)
    print("FINAL RESULTS (on YOUR validation set)")
    print("="*60)
    print(f"{'Method':<25} {'MAE':<10} {'Improvement':<12}")
    print("-" * 60)
    
    baseline_mae = results['Baseline']
    for method, mae in results.items():
        improvement = baseline_mae - mae
        marker = " ⭐" if mae == min(results.values()) else ""
        print(f"{method:<25} {mae:<10.3f} {improvement:+10.3f}{marker}")
    
    print("="*60)
    print(f"\nBest method: {min(results, key=results.get)}")
    print(f"Best MAE: {min(results.values()):.3f} months")
    print(f"Target (Paper H+R1): 5.4 months")
    print(f"Gap: {min(results.values()) - 5.4:.3f} months")
    print("="*60)
    
    # Save calibrators for later use
    print("\nSaving calibrators...")
    import pickle
    with open('gender_calibrator.pkl', 'wb') as f:
        pickle.dump(gender_cal, f)
    with open('isotonic_calibrator.pkl', 'wb') as f:
        pickle.dump(isotonic_cal, f)
    print("✓ Calibrators saved!")

if __name__ == "__main__":
    main()

Loading data...


KeyboardInterrupt: 